<a href="https://colab.research.google.com/github/bordeauxnate-oss/ThinkPythonAssignments/blob/main/Week15/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import json
from dataclasses import dataclass, field, asdict
from typing import List, Optional

# -----------------------------
# Helper Functions
# -----------------------------

def safe_int(prompt: str) -> int:
    """Safely get an integer from the user."""
    while True:
        value = input(prompt)
        try:
            return int(value)
        except ValueError:
            print("Please enter a valid number.")

def safe_choice(prompt: str, options: List[str]) -> str:
    """Ensure user selects a valid option."""
    while True:
        choice = input(prompt).strip().lower()
        if choice in options:
            return choice
        print(f"Invalid choice. Options: {', '.join(options)}")

def safe_nonempty(prompt: str) -> str:
    """Ensure user enters non-empty text."""
    while True:
        text = input(prompt).strip()
        if text:
            return text
        print("Input cannot be empty.")


# -----------------------------
# Data Model
# -----------------------------

@dataclass
class Combatant:
    name: str
    initiative: int
    hp: int
    ac: int
    type: str  # "player" or "monster"
    conditions: List[str] = field(default_factory=list)
    alive: bool = True

    def apply_damage(self, amount: int):
        self.hp -= amount
        if self.hp <= 0:
            self.hp = 0
            self.alive = False

    def heal(self, amount: int):
        if self.alive:
            self.hp += amount

    def add_condition(self, condition: str):
        if condition and condition not in self.conditions:
            self.conditions.append(condition)

    def remove_condition(self, condition: str):
        if condition in self.conditions:
            self.conditions.remove(condition)


# -----------------------------
# Combat Tracker Logic
# -----------------------------

class CombatTracker:
    def __init__(self):
        self.combatants: List[Combatant] = []
        self.turn_index: int = 0

    # -------------------------
    # Core Combat Functions
    # -------------------------

    def add_combatant(self, combatant: Combatant):
        self.combatants.append(combatant)
        self.sort_initiative()

    def sort_initiative(self):
        self.combatants.sort(key=lambda c: c.initiative, reverse=True)

    def next_turn(self):
        if not self.combatants:
            print("No combatants in the tracker.")
            return

        for _ in range(len(self.combatants)):
            self.turn_index = (self.turn_index + 1) % len(self.combatants)
            if self.combatants[self.turn_index].alive:
                break

    def current_turn(self) -> Optional[Combatant]:
        if not self.combatants:
            return None

        # Ensure turn index is valid
        if self.turn_index >= len(self.combatants):
            self.turn_index = 0

        return self.combatants[self.turn_index]

    def show_combatants(self):
        print("\n=== Combatants ===")
        for c in self.combatants:
            status = "DEFEATED" if not c.alive else f"HP: {c.hp}"
            conds = ", ".join(c.conditions) if c.conditions else "None"
            print(f"{c.name} | Init: {c.initiative} | AC: {c.ac} | {status} | Conditions: {conds}")
        print("==================\n")

    # -------------------------
    # Saving & Loading
    # -------------------------

    def save_to_file(self, filename="combat_data.json"):
        data = {
            "turn_index": self.turn_index,
            "combatants": [asdict(c) for c in self.combatants]
        }
        try:
            with open(filename, "w") as f:
                json.dump(data, f, indent=4)
            print(f"Combat saved to {filename}")
        except Exception as e:
            print(f"Error saving file: {e}")

    def load_from_file(self, filename="combat_data.json"):
        try:
            with open(filename, "r") as f:
                data = json.load(f)

            self.turn_index = data.get("turn_index", 0)
            self.combatants = []

            for cdata in data.get("combatants", []):
                try:
                    self.combatants.append(Combatant(**cdata))
                except TypeError:
                    print(f"Skipping invalid combatant entry: {cdata}")

            self.sort_initiative()

            # Fix invalid turn index
            if self.turn_index >= len(self.combatants):
                self.turn_index = 0

            print(f"Combat loaded from {filename}")

        except FileNotFoundError:
            print("No saved combat found.")
        except json.JSONDecodeError:
            print("Error: Saved file is corrupted.")
        except Exception as e:
            print(f"Unexpected error loading file: {e}")


# -----------------------------
# Menu Interface
# -----------------------------

def main_menu():
    tracker = CombatTracker()

    while True:
        print("=== D&D Combat Tracker ===")
        print("1. Add Combatant")
        print("2. Show Combatants")
        print("3. Advance Turn")
        print("4. Modify Combatant")
        print("5. Show Current Turn")
        print("6. Save Combat")
        print("7. Load Combat")
        print("8. Exit")

        choice = safe_choice("Choose an option: ",
                             ["1", "2", "3", "4", "5", "6", "7", "8"])

        if choice == "1":
            name = safe_nonempty("Name: ")
            initiative = safe_int("Initiative: ")
            hp = safe_int("HP: ")
            ac = safe_int("AC: ")
            ctype = safe_choice("Type (player/monster): ", ["player", "monster"])

            tracker.add_combatant(Combatant(name, initiative, hp, ac, ctype))
            print(f"{name} added.\n")

        elif choice == "2":
            tracker.show_combatants()

        elif choice == "3":
            tracker.next_turn()
            current = tracker.current_turn()
            if current:
                print(f"It is now {current.name}'s turn.\n")

        elif choice == "4":
            name = safe_nonempty("Enter combatant name: ")
            target = next((c for c in tracker.combatants if c.name == name), None)

            if not target:
                print("Combatant not found.\n")
                continue

            print("1. Damage")
            print("2. Heal")
            print("3. Add Condition")
            print("4. Remove Condition")

            action = safe_choice("Choose: ", ["1", "2", "3", "4"])

            if action == "1":
                dmg = safe_int("Damage amount: ")
                target.apply_damage(dmg)
                print(f"{target.name} now has {target.hp} HP.\n")

            elif action == "2":
                heal = safe_int("Heal amount: ")
                target.heal(heal)
                print(f"{target.name} now has {target.hp} HP.\n")

            elif action == "3":
                cond = safe_nonempty("Condition to add: ")
                target.add_condition(cond)
                print(f"Added condition to {target.name}.\n")

            elif action == "4":
                cond = safe_nonempty("Condition to remove: ")
                target.remove_condition(cond)
                print(f"Removed condition from {target.name}.\n")

        elif choice == "5":
            current = tracker.current_turn()
            if current:
                print(f"Current turn: {current.name} (HP: {current.hp})\n")
            else:
                print("No combatants yet.\n")

        elif choice == "6":
            tracker.save_to_file()

        elif choice == "7":
            tracker.load_from_file()

        elif choice == "8":
            print("Exiting combat tracker.")
            break


# Run the menu
main_menu()


=== D&D Combat Tracker ===
1. Add Combatant
2. Show Combatants
3. Advance Turn
4. Modify Combatant
5. Show Current Turn
6. Save Combat
7. Load Combat
8. Exit
Choose an option: 1
Name: Geralt
Initiative: ten
Please enter a valid number.
Initiative: 40
HP: 
Please enter a valid number.
HP: 32
AC: eight
Please enter a valid number.
AC: 18
Type (player/monster): none
Invalid choice. Options: player, monster
Type (player/monster): Player
Geralt added.

=== D&D Combat Tracker ===
1. Add Combatant
2. Show Combatants
3. Advance Turn
4. Modify Combatant
5. Show Current Turn
6. Save Combat
7. Load Combat
8. Exit
Choose an option: 1
Name: Goblin
Initiative: 15
HP: 7
AC: 15
Type (player/monster): monster
Goblin added.

=== D&D Combat Tracker ===
1. Add Combatant
2. Show Combatants
3. Advance Turn
4. Modify Combatant
5. Show Current Turn
6. Save Combat
7. Load Combat
8. Exit
Choose an option: 1
Name: Wolf
Initiative: 10
HP: 11
AC: 13
Type (player/monster): monster
Wolf added.

=== D&D Combat Tracke